##### Three Parts of This Lab

This lab uses the OpenAI Agents SDK and OpenAI client interface with Gemini's OpenAI-compatible endpoint. The Agents SDK's hosted OpenAI tracing is disabled for these Gemini calls.


###### Part 1: Building a Simple Agent Loop
Build a basic agent loop, then stream responses as they are generated. Since model requests go to Gemini, OpenAI-hosted tracing is disabled in this setup.

###### Part 2: Adding Tool Calls
Give the agent tools so it can take actions beyond generating text.

###### Part 3: Adding Memory
Add memory so the agent can retain information and use context from earlier interactions.

In [56]:
import os
import requests
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, set_tracing_disabled
from openai import AsyncOpenAI
from openai.types.responses import ResponseTextDeltaEvent


In [57]:
load_dotenv(override=True)

True

In [58]:
gemini_api_key= os.getenv("GEMINI_API_KEY")
gemini_base_url = os.getenv("GEMINI_BASE_URL")

gemini_client = AsyncOpenAI(
    api_key=gemini_api_key,
    base_url=gemini_base_url,
    timeout=60.0,
   
)
gemini_model= OpenAIChatCompletionsModel(
              model="gemini-3.5-flash-lite",
              openai_client= gemini_client
           )



In [59]:

# Build the agent with Gemini served through the OpenAI-compatible Chat Completions API.
agent = Agent(name="Jokester", instructions="Tell clear, family-friendly jokes.", model=gemini_model)


In [60]:
# Run the Gemini-backed agent without sending traces to OpenAI.
result = await Runner.run(agent, "Tell me a short joke about autonomous AI agents.")


In [61]:
#There is final output
print(result.final_output)

Why did the autonomous AI agent refuse to go to sleep?

It was afraid of missing the latest patch notes!


In [62]:
#Here is the details of LLM calls
result.to_input_list()

[{'content': 'Tell me a short joke about autonomous AI agents.',
  'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Why did the autonomous AI agent refuse to go to sleep?\n\nIt was afraid of missing the latest patch notes!',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gemini-3.5-flash-lite',
   'response_id': 'MDzgapK2AaDl4t4P95jL8As'}}]

##### Adding Observability with trace()
trace(): Records everything that happens inside an agent run for debugging and monitoring.

###### It logs:
LLM calls,
Tool calls,
Handoffs,
Guardrails,
Execution time,

all under one trace

In [64]:
with trace("Your Joke is ready"):
    result= await Runner.run(agent, "tell me a joke about human")

print(result.final_output)

Why are humans so bad at playing hide and seek?

Because good humans are hard to find!


##### Streaming -> Runner.run_streamed()
Streaming is the process of delivering an agent's output incrementally, including 
###### generated tokens 
###### workflow events
 as they become available instead of waiting for the complete execution to finish

In [55]:
result = Runner.run_streamed(agent, " tell me 5 jokes about ai agents")
async for event in result.stream_events():
    if event.type=="raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)


Here are 5 family-friendly jokes about AI agents:

1. **Why did the AI agent cross the road?**
   Because the training data said so!

2. **What is an AI agent’s favorite type of music?**
   Algorithmic and blues!

3. **Why did the AI agent get locked out of its house?**
   It lost its master key… and its backup key, and its cloud backup, and its cache!

4. **How do AI agents like to drink their coffee?**
   Java-based, with a lot of processing power!

5. **Why did the AI agent go to art school?**
   It wanted to learn how to draw better conclusions!